# Fine-tune SHADOW (LoRA, free Colab GPU)

Turns qwen3:1.7b into a **butler-trained** model: it learns tone and phrasing from his real chat logs (`chat_log_export.jsonl`) plus the seed `butler_examples.jsonl`.

**How to use:**
1. Runtime > Change runtime type > **T4 GPU** (free tier works)
2. Run each cell in order
3. Upload `chat_log_export.jsonl` and `butler_examples.jsonl` when the file cell asks (or skip upload to train on the seed examples only)
4. At the end, download `shadow-lora/` and run the local import cell on your laptop

Time: ~20-40 min on a free T4. No paid account needed.

## 1. Install training stack

In [ ]:
%pip install -q unsloth
%pip install -q --force-reinstall --no-deps git+https://github.com/unslothai/unsloth.git

## 2. Upload your datasets
Skip the upload (press Enter/cancel the picker) to train on the seed butler examples only - you can still proceed.

In [ ]:
import os
from google.colab import files

SEED = "butler_examples.jsonl"
REAL = "chat_log_export.jsonl"

if not os.path.exists(SEED):
    print("Upload butler_examples.jsonl (from the repo's training/ folder):")
    up = files.upload()

if not os.path.exists(REAL):
    print("(Optional) Upload chat_log_export.jsonl - cancel to train on seed data only:")
    try:
        up = files.upload()
    except Exception:
        pass

print("Have:", [f for f in (SEED, REAL) if os.path.exists(f)])

## 3. Load the base model (4-bit, T4-friendly)

In [ ]:
from unsloth import FastLanguageModel

MODEL = "unsloth/Qwen3-1.7B"
MAX_SEQ = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

## 4. Build the training set
Mixes real chats (weight 3x - your butler's actual voice) with seed examples.

In [ ]:
import json, random
from datasets import Dataset

def load_jsonl(path):
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

seed = load_jsonl(SEED)
real = load_jsonl(REAL)

# Real chats matter most: repeat them 3x.
rows = seed + real * 3
random.seed(42); random.shuffle(rows)
print(f"seed={len(seed)} real={len(real)} -> training rows={len(rows)}")

def to_text(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False)}

dataset = Dataset.from_list(rows).map(to_text)

## 5. Train (LoRA)
Small on purpose: a 1.7B model overfits fast on tiny datasets. Stop early if loss flattens.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        output_dir="shadow_out",
        report_to="none",
        seed=42,
    ),
)

trainer.train()

## 6. Chat with the trained butler (sanity check)

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are SHADOW, the user's personal AI butler - a refined British gentleman with dry wit who calls the user 'sir'."},
    {"role": "user", "content": "I stayed up all night debugging again."},
]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
out = model.generate(inputs, max_new_tokens=80, temperature=0.7)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

## 7. Export the LoRA adapter
Download the produced `shadow-lora.zip` - that small file carries everything he learned.

In [ ]:
model.save_pretrained("shadow-lora")
tokenizer.save_pretrained("shadow-lora")

!zip -r shadow-lora.zip shadow-lora
from google.colab import files
files.download("shadow-lora.zip")

## 8. LOCAL import (run on the LAPTOP, not in Colab)
Unzip `shadow-lora.zip` into `C:\Users\us448\JARVIS\training\`, then run this cell **locally** (plain Python, needs `pip install llama-cpp-python` OR the ollama import path below).

Simplest local path - merge with Ollama's own tooling:
```bash
# one-time: pip install llama-cpp-python  (or use ollama's upcoming import)
python training/import_lora.py
```